# Non Gaussian Likelihoods

$$
$$

<!-- CIP-0012 Track A: no-op outside HTML (overridden in talk-macros-html.gpp). -->
<!-- Class-test timer: no-op outside slides HTML (overridden in talk-macros-slides-html.gpp). -->
<!-- CIP-0016: mdpp expands \diagramcredit before gpp; leftover calls are empty. -->
<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!---->
<!-- CIP-0012 Track A: content script load. Once-guard with alphanumeric
     names stays in the surrounding ifndef wrapper (GPP ifndef limitation). -->
<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!-- The last names to be defined. Should be defined entirely in terms of macros from above-->
<!--

-->

## Gaussian Process Summer School notebook setup

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gpss/includes/gpss-notebook-setup.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gpss/includes/gpss-notebook-setup.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

We install packages used across the notebooks for data loading, plotting
helpers, and Gaussian process modelling.

In [ ]:
import importlib.util

In [ ]:
def install_command(imp):
    import shutil
    import sys
    packages = {
        'daft': 'daft-pgm',
        'mlai': 'git+https://github.com/lawrennd/mlai.git',
        'pods': 'pods',
        'PyDeepGP': 'git+https://github.com/SheffieldML/PyDeepGP.git',
        'notutils': 'git+https://github.com/lawrennd/notutils.git',
        'pods' : 'git+https://github.com/lawrennd/ods.git',
        'lamd': 'lamd',
        'GPy': 'gpy>=1.15',
    'emukit': 'emukit',
    }

    def _installer():
        if shutil.which('uv'):
            return 'uv pip install --python "' + sys.executable + '"'
        return '"' + sys.executable + '" -m pip install'

    def _ver_tuple(v):
        parts = []
        for p in v.split('.')[:3]:
            try:
                parts.append(int(''.join(ch for ch in p if ch.isdigit()) or '0'))
            except ValueError:
                parts.append(0)
        while len(parts) < 3:
            parts.append(0)
        return tuple(parts)

    def _below_floor(spec):
        if '>=' not in spec:
            return False
        name, floor = spec.split('>=', 1)
        name = name.strip()
        floor = floor.strip().split(',')[0].strip()
        try:
            import importlib.metadata as md
            cur = md.version(name)
        except Exception:
            return True
        return _ver_tuple(cur) < _ver_tuple(floor)

    spec = packages[imp] if imp in packages else imp
    missing = importlib.util.find_spec(imp) is None
    if missing or _below_floor(spec):
        return _installer() + ' "' + spec + '"'
    return 'echo "Package ' + imp + ' is already installed."'

In [ ]:
cmd = install_command('pods')

In [ ]:
%system {cmd}

In [ ]:
cmd = install_command('notutils')

In [ ]:
%system {cmd}

In [ ]:
cmd = install_command('mlai')

In [ ]:
%system {cmd}

In [ ]:
import matplotlib.pyplot as plt
import shutil

if shutil.which('latex') is None:
    plt.rcParams['text.usetex'] = False
else:
    plt.rcParams['text.usetex'] = True
    plt.rcParams['text.latex.preamble'] = (
        r'\usepackage{amsmath}\usepackage{amssymb}\usepackage{mathrsfs}'
    )

plt.rcParams.update({'font.size': 22})

<!--setupplotcode{import seaborn as sns
sns.set_style('darkgrid')
sns.set_context('paper')
sns.set_palette('colorblind')}-->

In [ ]:
cmd = install_command('GPy')

In [ ]:
%system {cmd}

## GPy: A Gaussian Process Framework in Python

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_software/includes/gpy-software.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_software/includes/gpy-software.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Gaussian processes are a flexible tool for non-parametric analysis with
uncertainty. The GPy software was started in Sheffield to provide a easy
to use interface to GPs. One which allowed the user to focus on the
modelling rather than the mathematics.

<img class="" src="https://mlatcl.github.io/gpss/slides/diagrams/gp/gpy.png" style="width:70%; height:auto; display:block;">

Figure: <i>GPy is a BSD licensed software code base for implementing
Gaussian process models in Python. It is designed for teaching and
modelling. We welcome contributions which can be made through the GitHub
repository <https://github.com/SheffieldML/GPy></i>

GPy is a BSD licensed software code base for implementing Gaussian
process models in python. This allows GPs to be combined with a wide
variety of software libraries.

The software itself is available on
[GitHub](https://github.com/SheffieldML/GPy) and the team welcomes
contributions.

The aim for GPy is to be a probabilistic-style programming language,
i.e., you specify the model rather than the algorithm. As well as a
large range of covariance functions the software allows for non-Gaussian
likelihoods, multivariate outputs, dimensionality reduction and
approximations for larger data sets.

The documentation for GPy can be found
[here](https://gpy.readthedocs.io/en/latest/).

# Non Gaussian Likelihoods

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gp/includes/non-gaussian-gps.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gp/includes/non-gaussian-gps.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

\[Gaussian processes model functions. If our observation is a corrupted
version of this function and the corruption process is *also* Gaussian,
it is trivial to account for this. However, there are many circumstances
where our observation may be non Gaussian. In these cases we need to
turn to approximate inference techniques. As a simple illustration,
we’ll use a dataset of binary observations of the language that is
spoken in different regions of East-Timor. First we will load the data
and a couple of libraries to visualize it.}

In [ ]:
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
import pods

In [ ]:
data = pods.datasets.east_timor()
X = data['X']
y = data['Y']
polygons = data['polygons']

Now we will create a map of East Timor and, using GPy, plot the data on
top of it. A classification model can be defined in a similar way to the
regression model, but now using `GPy.models.GPClassification`. However,
once we’ve define the model, we also need to update the approximation to
the likelihood. This runs the Expectation propagation updates.

In [ ]:
#Visualize a map of East-Timor
fig = plt.figure()
ax = fig.add_subplot(111)
for p in polygons:
    ax.add_collection(PatchCollection([Polygon(p)],facecolor="#F4A460"))
ax.set_xlim(124.,127.5)
ax.set_ylim(-9.6,-8.1)
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")

In [ ]:
import GPy

In [ ]:
#Define the model
kern = GPy.kern.RBF(2)
m = GPy.models.GPClassification(X,y, kernel=kern)
print(m)

In [ ]:
m.plot(ax=ax)

The decision boundary should be quite poor! However we haven’t optimized
the model. Try the following:

In [ ]:
m.optimize()
print(m)

In [ ]:
m.plot()

The optimization is based on the likelihood approximation that was made
after we constructed the model. However, because we’ve now changed the
model parameters the quality of that approximation has now probably
deteriorated. To improve the model we should iterate between updating
the Expectation propagation approximation and optimizing the model
parameters.

## Olympic Marathon Data

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_datasets/includes/olympic-marathon-data.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_datasets/includes/olympic-marathon-data.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

-   Gold medal times for Olympic Marathon since 1896.
-   Marathons before 1924 didn’t have a standardized distance.
-   Present results using pace per km.
-   In 1904 Marathon was badly organized leading to very slow times.

<img class="" src="https://mlatcl.github.io/gpss/slides/diagrams/datasets/eliud-kipchoge_berlin_2015.jpg" style="width:100%; height:auto; display:block;">
<small>Image from [Wikimedia
Commons](https://commons.wikimedia.org/wiki/File:Eliud_Kipchoge_in_Berlin_-_2015_(cropped).jpg)</small>

The Olympic Marathon data is a standard dataset for regression
modelling. The data consists of the pace of Olympic Gold Medal Marathon
winners for the Olympics from 1896 to present. Let’s load in the data
and plot.

In [ ]:
cmd = install_command('pods')

In [ ]:
%system {cmd}

In [ ]:
import numpy as np
import pods

In [ ]:
data = pods.datasets.olympic_marathon_men()
x = data['X']
y = data['Y']

offset = y.mean()
scale = np.sqrt(y.var())
yhat = (y - offset)/scale

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai

In [ ]:

xlim = (1876,2044)
ylim = (2.5, 6.5)

fig, ax = plt.subplots(figsize=plot.big_wide_figsize)
_ = ax.plot(x, y, 'r.',markersize=10)
ax.set_xlabel('year', fontsize=20)
ax.set_ylabel('pace min/km', fontsize=20)
ax.set_xlim(xlim)
ax.set_ylim(ylim)

mlai.write_figure(filename='olympic-marathon.svg', 
                  directory='./datasets')

<img src="https://mlatcl.github.io/gpss/slides/diagrams/datasets/olympic-marathon.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Olympic marathon pace times since 1896.</i>

Things to notice about the data include the outlier in 1904, in that
year the Olympics was in St Louis, USA. Organizational problems and
challenges with dust kicked up by the cars following the race meant that
participants got lost, and only very few participants completed. More
recent years see more consistently quick marathons.

## Alan Turing

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/alan-turing-marathon.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/alan-turing-marathon.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

<img class="" src="https://mlatcl.github.io/gpss/slides/diagrams/turing-times.gif" style="width:100%; height:auto; display:block;">

<img class="" src="https://mlatcl.github.io/gpss/slides/diagrams/turing-run.jpg" style="width:50%; height:auto; display:block;">

Figure: <i>Alan Turing, in 1946 he was only 11 minutes slower than the
winner of the 1948 games. Would he have won a hypothetical games held in
1946? Source:
<a href="http://www.turing.org.uk/scrapbook/run.html" target="_blank">Alan
Turing Internet Scrapbook</a>.</i>

If we had to summarise the objectives of machine learning in one word, a
very good candidate for that word would be *generalization*. What is
generalization? From a human perspective it might be summarised as the
ability to take lessons learned in one domain and apply them to another
domain. If we accept the definition given in the first session for
machine learning, $$
\text{data} + \text{model} \stackrel{\text{compute}}{\rightarrow} \text{prediction}
$$ then we see that without a model we can’t generalise: we only have
data. Data is fine for answering very specific questions, like “Who won
the Olympic Marathon in 2012?”, because we have that answer stored,
however, we are not given the answer to many other questions. For
example, Alan Turing was a formidable marathon runner, in 1946 he ran a
time 2 hours 46 minutes (just under four minutes per kilometer, faster
than I and most of the other [Endcliffe Park
Run](http://www.parkrun.org.uk/sheffieldhallam/) runners can do 5 km).
What is the probability he would have won an Olympics if one had been
held in 1946?

To answer this question we need to generalize, but before we formalize
the concept of generalization let’s introduce some formal representation
of what it means to generalize in machine learning.

## Gaussian Process Fit

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gp/includes/gp-fit.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gp/includes/gp-fit.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Our first objective will be to perform a Gaussian process fit to the
data, we’ll do this using the [GPy
software](https://github.com/SheffieldML/GPy).

In [ ]:
import GPy

In [ ]:
m_full = GPy.models.GPRegression(x,yhat)
_ = m_full.optimize() # Optimize parameters of covariance function

The first command sets up the model, then `m_full.optimize()` optimizes
the parameters of the covariance function and the noise level of the
model. Once the fit is complete, we’ll try creating some test points,
and computing the output of the GP model in terms of the mean and
standard deviation of the posterior functions between 1870 and 2030. We
plot the mean function and the standard deviation at 200 locations. We
can obtain the predictions using `y_mean, y_var = m_full.predict(xt)`

In [ ]:
xt = np.linspace(1870,2030,200)[:,np.newaxis]
yt_mean, yt_var = m_full.predict(xt)
yt_sd=np.sqrt(yt_var)

Now we plot the results using the helper function in `mlai.plot`.

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_wide_figsize)
plot.model_output(m_full, scale=scale, offset=offset, ax=ax, xlabel="year", ylabel="pace min/km", fontsize=20, portion=0.2)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
mlai.write_figure(figure=fig,
                  filename="olympic-marathon-gp.svg",
                  directory = "./gp",
                  transparent=True, frameon=True)

<img src="https://mlatcl.github.io/gpss/slides/diagrams/gp/olympic-marathon-gp.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Gaussian process fit to the Olympic Marathon data. The error
bars are too large, perhaps due to the outlier from 1904.</i>

## Fit Quality

In the fit we see that the error bars (coming mainly from the noise
variance) are quite large. This is likely due to the outlier point in
1904, ignoring that point we can see that a tighter fit is obtained. To
see this make a version of the model, `m_clean`, where that point is
removed.

In [ ]:
x_clean=np.vstack((x[0:2, :], x[3:, :]))
y_clean=np.vstack((yhat[0:2, :], yhat[3:, :]))

m_clean = GPy.models.GPRegression(x_clean,y_clean)
_ = m_clean.optimize()

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_wide_figsize)
plot.model_output(m_clean, scale=scale, offset=offset, ax=ax, xlabel='year', ylabel='pace min/km', fontsize=20, portion=0.2)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
mlai.write_figure(figure=fig,
                  filename='./gp/olympic-marathon-gp-outlier-removed.svg', 
                  transparent=True, frameon=True)

<img src="https://mlatcl.github.io/gpss/slides/diagrams/gp/olympic-marathon-gp-outlier-removed.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Gaussian process fit to the Olympic Marathon data. Here we’ve
removed the outlier from 1904.</i>

## Robust Regression: A Running Example

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gp/includes/olympic-marathon-gp-student-t.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gp/includes/olympic-marathon-gp-student-t.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

We already considered the olympic marathon data. In 1904 we noted there
was an outlier example. Today we’ll see if we can deal with that outlier
by considering a non-Gaussian likelihood. Noise sampled from a
Student-$t$ density is heavier tailed than that sampled from a Gaussian.
However, it cannot be trivially assimilated into the Gaussian process.
Below we use the *Laplace approximation* to incorporate this noise
model.

In [ ]:
GPy.likelihoods.StudentT?

In [ ]:
# make a student t likelihood with standard parameters
t_distribution = GPy.likelihoods.StudentT(deg_free=3, sigma2=2)
laplace = GPy.inference.latent_function_inference.Laplace()

kern = GPy.kern.RBF(1, lengthscale=5) + GPy.kern.Bias(1)
model = GPy.core.GP(x, y, kernel=kern, inference_method=laplace, likelihood=t_distribution)
model.constrain_positive('t_noise')

model.optimize(messages=True)
display(model)

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_wide_figsize)
# Plot latent function (Student-$t$ predictive moments currently fail in GPy under NumPy/SciPy).
model.plot(ax=ax, plot_raw=True)

mlai.write_figure('olympic-marathon-gp-student-t.svg', directory='./gp')

<img src="https://mlatcl.github.io/gpss/slides/diagrams/gp/olympic-marathon-gp-student-t.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Fit to the Olympic marathon data using the Student-$t$
likelihood and the Laplace approximation.</i>

# Sparse GP Classification

In this section we’ll combine expectation propagation with the low rank
approximation to build a simple image classification application. For
this toy example we’ll classify whether or not the subject of the image
is wearing glasses.

Correspond to whether the subject of the image is wearing glasses. Set
up the ipython environment and download the data:

In [ ]:
from scipy import io

First let’s retrieve some data. We will use the ORL faces data set, our
objective will be to classify whether or not the subject in an image is
wearing glasess.

Here’s a simple way to visualise the data. Each pixel in the image will
become an input to the GP.

In [ ]:
import pods

In [ ]:
data = pods.datasets.olivetti_glasses()
X = data['X']
y = data['Y']
Xtest = data['Xtest']
ytest = data['Ytest']
print(data['info'], data['details'], data['citation'])

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_figsize)
ax.imshow(X[0].reshape(64, 64, order='F'),
          interpolation='nearest',
          cmap=plt.cm.gray)

mlai.write_figure("olivetti-glasses-image.png", 
                  directory="./datasets")

<img class="" src="https://mlatcl.github.io/gpss/slides/diagrams/datasets/olivetti-glasses-image.png" style="width:60%; height:auto; display:block;">

Figure: <i>Image from the Oivetti glasses data sets.</i>

Next we choose some inducing inputs. Here we’ve chosen inducing inputs
by applying k-means clustering to the training data. Think about whether
this is a good scheme for choosing the inputs? Can you devise a better
one?

In [ ]:
X.shape
y.shape
print(X)

In [ ]:
from scipy import cluster

In [ ]:
M = 8
X = (X - X.mean(0)[np.newaxis,:])/X.std(0)[np.newaxis,:]
Z = np.random.permutation(X)[:M]

Finally, we’re ready to build the classifier object.

In [ ]:
kern = GPy.kern.RBF(X.shape[1],lengthscale=20) + GPy.kern.White(X.shape[1],0.001)
model = GPy.models.SparseGPClassification(X, y, kernel=kern, Z=Z)
model.optimize(messages=True)
display(model)

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_figsize)
ax.imshow(model.Z.gradient[0].reshape(64,64,order='F'),interpolation='nearest',cmap=plt.cm.gray)

mlai.write_figure('olivetti-inducing-variable-gradients.png', directory='./gp')

<img class="" src="https://mlatcl.github.io/gpss/slides/diagrams/gp/olivetti-inducing-variable-gradients.png" style="width:60%; height:auto; display:block;">

Figure: <i>The gradients of the inducing variable.</i>

## Thanks!

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/mlatcl/gpss/edit/gh-pages/_lamd/non-gaussian-likelihoods.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/mlatcl/gpss/edit/gh-pages/_lamd/non-gaussian-likelihoods.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

For more information on these subjects and more you might want to check
the following resources.

-   company: [Trent AI](https://trent.ai)
-   book: [The Atomic
    Human](https://www.penguin.co.uk/books/455130/the-atomic-human-by-lawrence-neil-d/9780241625248)
-   twitter: [@lawrennd](https://twitter.com/lawrennd)
-   podcast: [The Talking Machines](http://thetalkingmachines.com)
-   newspaper: [Guardian Profile
    Page](http://www.theguardian.com/profile/neil-lawrence)
-   blog:
    [http://inverseprobability.com](http://inverseprobability.com/blog.html)

## References